# Documentation for the Jupyter Notebook

This Jupyter Notebook processes APCEMM output files to extract and prepare data for training or validation of a machine learning model. The notebook reads netCDF files, extracts relevant data, processes it, and saves the processed data into a `.npy` file.

## Imports
This cell imports the necessary libraries for the notebook:
- `os.path` for file path manipulations.
- `xarray` for handling netCDF files.
- `numpy` for numerical operations.

## Pre-processing APCEMM outputs
Defines a class `apce_data_struct` to store the extracted data and a function `read_apcemm_data` to read and process APCEMM output files from a specified directory. The function extracts time, integrated optical depth (`int_OD`), and relative humidity (`RHi`) data from the netCDF files.

Initializes variables and sets parameters for processing the data:
- `training_sample_matrix_output` and `training_sample_matrix_input` to store the processed data.
- `scaled_mean` and `scaling_factor` for normalizing the data.
- `test`, `num_runs`, and `validation` to specify the test run number, number of runs, and whether the data is for validation or training.

## Save APCEMM output variables of interest
Loops through the specified number of runs, reads the APCEMM output data and corresponding relative humidity data, processes the data, and appends it to the respective matrices. The data is normalized and reshaped as required.

## Saving data
Saves the processed data (`training_sample_matrix`) into a `.npy` file for later use in training or validating the machine learning model.

In [16]:
#Import Libs
import os.path
import xarray as xr
import numpy as np

In [17]:
#Functions that will be used for postprocessing
class apce_data_struct:
    def __init__(self, t, ds_t, int_OD, RHi):
        self.t = t
        self.ds_t = ds_t
        self.int_OD = int_OD
        self.RHi = RHi
    
def read_apcemm_data(directory):
    t_mins = []
    ds_t = []
    int_OD = []
    RHi = []

    for file in sorted(os.listdir(directory)):
        if(file.startswith('ts_aerosol') and file.endswith('.nc')):
            file_path = os.path.join(directory,file)
            ds = xr.open_dataset(file_path, engine = "netcdf4", decode_times = False)
            ds_t.append(ds)
            tokens = file_path.split('.')
            mins = int(tokens[-2][-2:])
            hrs = int(tokens[-2][-4:-2])
            t_mins.append(hrs*60 + mins)
            int_OD.append(ds["intOD"])
            RHi.append(ds["RHi"])

    return apce_data_struct(t_mins, ds_t, int_OD, RHi)

In [18]:
# Extract integrated vertical optical depth data from APCEMM output files
sample_matrix_output = []
sample_matrix_input = []
sample_arrays = []
scaled_mean = 117 # Reducing back to [0,1] space after APCEMM scaling

test_id = "test_7" # Test run number
num_runs = 20 # Number of test runs
validation = False # Are you processing a validation set? If False assumes training set

if validation == True: # If you are processing a validation set
    run_type = "validation"
else: # If you are processing a training set
    run_type = "training"

In [19]:
for i in range(1, num_runs+1): # For test runs num_runs
    if validation == True: # If you are processing a validation set
        apce_data = read_apcemm_data(f'/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/{test_id}/outputs/validation/{test_id}_run_{i}')
        input_RHi_ds = xr.open_dataset(f'/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/{test_id}/inputs/validation/APCEMM_met_validation_{i}.nc')
    else: # If you are processing a training set
        apce_data = read_apcemm_data(f'/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/{test_id}/outputs/training/{test_id}_run_{i}')
        input_RHi_ds = xr.open_dataset(f'/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/{test_id}/inputs/training/APCEMM_met_{i}.nc')
    
    int_OD = apce_data.int_OD
    recorded_timesteps = len(int_OD)

    # Normalizing outputs to a 12 hour timeframe
    if recorded_timesteps == 0: # Ignore any runs where a contrail does not form
        continue
    elif recorded_timesteps > 73: # Truncate to 73 timesteps if contrail persists longer than 12 hours
        current_sample_output = np.array(int_OD)[:73].reshape(1, 73)[0] # 1 row, 73 columns
    elif recorded_timesteps < 73: # Pad with zeros to reach 73 timesteps
        current_sample_output = np.pad(np.array(int_OD).reshape(1, recorded_timesteps)[0], (0, 73 - recorded_timesteps), 'constant', constant_values = 0)
    
    # Define your input_RHi array (length 24 --> 24 hours in original met file)
    input_RHi = input_RHi_ds['relative_humidity_ice'][98].values # RHi values for times 0-24 hours at pressure level index 98. 
    
    # Expand input_RHi to match the timestamps output by APCEMM. APCEMM is run for 12 hours at 10-minute intervals. The met file is for 24 hours at 1-hour intervals.
    # 1 hour is 6 10-minute intervals, so we repeat each RHi value 6 times to match met input to the APCEMM output.
    expanded_input_RHi = []
    expanded_input_RHi.append(input_RHi[0]) # Count the zeroth timestep as a 7th repeat
    repeated_input_RHi = np.repeat(input_RHi[0:12], 6) # Look only at the first 12 hours of the met file. Repeat each element 6 times: [100 110 105] --> [100 100 100 100 100 100 110 110 110 110 110 110 105 105 105 105 105 105]
    expanded_input_RHi.extend(repeated_input_RHi)
    normalized_input_RHi = (np.array(expanded_input_RHi) - scaled_mean) # Shift the mean back to 0 for PCE to parse
    current_sample_input = normalized_input_RHi.reshape(1,73)[0] # Transform into a row vector for stacking
    
    # Stacking the input and output arrays
    stacked = np.column_stack((current_sample_input, current_sample_output)) # Shape: (73, 2) --> (column, depth) --> (timesteps, 1 input 1 output) INPUTS: (:,0), OUTPUTS: (:,1)
    sample_arrays.append(stacked)
    input_RHi_ds.close()

# This matrix will be used for training the machine learning model, it contains input RHi and output int_OD. In The future there will be multiple input variables.
sample_matrix = np.array(sample_arrays) # Shape: (num_runs, 73, 2) --> (depth, row, column) --> (number of datasets to train PCE, timesteps, 1 input 1 output) INPUTS: (:,:,0), OUTPUTS: (:,:,1)
# Transpose the matrix to match the expected input shape of the machine learning model
sample_matrix = np.transpose(sample_matrix, (2, 0, 1)) # Shape: (2, num_runs, 73) --> (depth, row, column) --> (1 input 1 output, number of datasets to train PCE, timesteps) INPUTS: (0,:,:), OUTPUTS: (1,:,:)
print(sample_matrix.shape)

(2, 20, 73)


In [20]:
# Save the training_sample_matrix to a .npy file
np.save('/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/{}/outputs/{}/{}_sample_matrix.npy'.format(test_id, run_type, run_type), sample_matrix)